In [1]:
!pip install -q torch torchvision pandas


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, Subset

In [3]:
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

PyTorch version: 2.13.0+cpu
Device: cpu


In [4]:
DATA_DIR = "./data"

TRAIN_SIZE = 5000
TEST_SIZE = 1000

BATCH_SIZE = 64

EPOCHS = 2

LEARNING_RATE = 0.001

In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [6]:
full_train_dataset = datasets.FashionMNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

full_test_dataset = datasets.FashionMNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

print("Original training images:", len(full_train_dataset))
print("Original testing images:", len(full_test_dataset))

100.0%
100.0%
100.0%
100.0%

Original training images: 60000
Original testing images: 10000


In [7]:
train_dataset = Subset(
    full_train_dataset,
    range(TRAIN_SIZE)
)

test_dataset = Subset(
    full_test_dataset,
    range(TEST_SIZE)
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

Training images: 5000
Testing images: 1000


In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Training batches:", len(train_loader))
print("Testing batches:", len(test_loader))

Training batches: 79
Testing batches: 16


In [9]:
def create_model(model_name):

    if model_name == "AlexNet":

        model = models.alexnet(
            weights=models.AlexNet_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            10
        )

    elif model_name == "VGG16":

        model = models.vgg16(
            weights=models.VGG16_Weights.DEFAULT
        )

        model.classifier[6] = nn.Linear(
            model.classifier[6].in_features,
            10
        )

    elif model_name == "ResNet50":

        model = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            10
        )

    else:

        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return model.to(device)

In [10]:
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
    )

In [11]:
def calculate_flops(model):

    model.eval()

    dummy_input = torch.randn(
        1,
        3,
        224,
        224
    ).to(device)

    with torch.profiler.profile(
        activities=[
            torch.profiler.ProfilerActivity.CPU
        ],
        with_flops=True
    ) as prof:

        with torch.no_grad():
            model(dummy_input)

    total_flops = 0

    for event in prof.key_averages():

        if event.flops is not None:
            total_flops += event.flops

    return total_flops

In [12]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total * 100
    )

    return epoch_loss, epoch_accuracy

In [13]:
@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    running_loss = 0.0

    correct = 0
    total = 0

    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    test_loss = (
        running_loss / total
    )

    test_accuracy = (
        correct / total * 100
    )

    return test_loss, test_accuracy

In [14]:
model_names = [
    "AlexNet",
    "VGG16",
    "ResNet50"
]

results = []

In [15]:
for model_name in model_names:

    print()
    print("=" * 60)
    print(model_name)
    print("=" * 60)

    # ==========================================
    # Create model
    # ==========================================

    model = create_model(model_name)

    # ==========================================
    # Parameters
    # ==========================================

    total_params = count_parameters(model)

    print(
        f"Parameters: "
        f"{total_params / 1e6:.2f} M"
    )

    # ==========================================
    # FLOPs
    # ==========================================

    print("Calculating FLOPs...")

    flops = calculate_flops(model)

    print(
        f"FLOPs: "
        f"{flops / 1e9:.2f} G"
    )

    # ==========================================
    # Optimizer
    # ==========================================

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    criterion = nn.CrossEntropyLoss()

    # ==========================================
    # Training
    # ==========================================

    start_time = time.time()

    for epoch in range(EPOCHS):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion
        )

        print(
            f"Epoch {epoch + 1}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_acc:.2f}%"
        )

    # ==========================================
    # Training Time
    # ==========================================

    training_time = (
        time.time()
        - start_time
    )

    # ==========================================
    # Test
    # ==========================================

    test_loss, test_acc = evaluate(
        model,
        test_loader
    )

    print(
        f"Test Loss: "
        f"{test_loss:.4f}"
    )

    print(
        f"Test Accuracy: "
        f"{test_acc:.2f}%"
    )

    print(
        f"Training Time: "
        f"{training_time:.2f} seconds"
    )

    # ==========================================
    # Save result
    # ==========================================

    results.append({

        "Model": model_name,

        "Accuracy (%)":
            round(test_acc, 2),

        "Loss":
            round(test_loss, 4),

        "Parameters (M)":
            round(
                total_params / 1e6,
                2
            ),

        "FLOPs (G)":
            round(
                flops / 1e9,
                2
            ),

        "Training Time (s)":
            round(
                training_time,
                2
            )
    })

    # ==========================================
    # Delete model
    # ==========================================

    del model

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


AlexNet
Parameters: 57.04 M
Calculating FLOPs...
FLOPs: 1.42 G
Epoch 1/2 | Train Loss: 1.2368 | Train Accuracy: 55.62%
Epoch 2/2 | Train Loss: 0.6736 | Train Accuracy: 76.10%
Test Loss: 0.6200
Test Accuracy: 76.50%
Training Time: 320.62 seconds

VGG16
Parameters: 134.30 M
Calculating FLOPs...
FLOPs: 30.93 G
Epoch 1/2 | Train Loss: 1.5573 | Train Accuracy: 48.38%
Epoch 2/2 | Train Loss: 0.6033 | Train Accuracy: 78.68%
Test Loss: 0.5718
Test Accuracy: 78.20%
Training Time: 6556.92 seconds

ResNet50
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\WINDOWS 11/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100.0%


Parameters: 23.53 M
Calculating FLOPs...
FLOPs: 8.17 G
Epoch 1/2 | Train Loss: 0.6369 | Train Accuracy: 77.84%
Epoch 2/2 | Train Loss: 0.3308 | Train Accuracy: 87.92%
Test Loss: 0.5028
Test Accuracy: 84.30%
Training Time: 2432.23 seconds


In [16]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy (%),Loss,Parameters (M),FLOPs (G),Training Time (s)
0,AlexNet,76.5,0.6200,57.04,1.42,320.62
1,VGG16,78.2,0.5718,134.30,30.93,6556.92
2,ResNet50,84.3,0.5028,23.53,8.17,2432.23


In [17]:
results_df.to_csv(
    "model_comparison.csv",
    index=False
)

print(
    "Saved to model_comparison.csv"
)

Saved to model_comparison.csv
